# 04 · Gap-tolerance LME sensitivity analysis

This notebook compares phase-level hypnodensity linear mixed-effects models across six persisted gap tolerances. The 90-second run is the primary analysis; all other tolerances are sensitivity analyses. No figures are generated.

## 1. Imports and parameters

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'outputs').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
RESULTS_DIR = OUTPUT_ROOT / 'gap_tolerance_lme_v1'
if RESULTS_DIR.exists():
    raise FileExistsError(f'{RESULTS_DIR} already exists; refusing to overwrite it.')

TOLERANCES = (0, 15, 30, 60, 90, 120)
STAGER = 'gssc'  # persisted label for GSSC
STAGES = ('W', 'N1', 'N2', 'N3', 'R')
PHASES = ('before', 'after', 'late')
CONDITIONS = ('placebo', 'DMT')
PROBABILITIES = tuple(f'prob_{stage}' for stage in STAGES)
CONTRAST_LABELS = {
    'after': 'After difference-in-differences',
    'late': 'Late difference-in-differences',
}
RUN_DIRS = {
    tolerance: OUTPUT_ROOT / f'gap_sensitivity_{tolerance:03d}s_gssc_yasa_sleepfm_cpu_v1'
    for tolerance in TOLERANCES
}
MODEL_FORMULA = (
    'phase_mean ~ C(condition, Treatment(reference=\"placebo\")) * '
    'C(phase, Treatment(reference=\"before\"))'
)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 30)

## 2. Statistical model description

Epochs are averaged within each `subject × condition × phase × electrode × stager` cell so that the model's observation is a phase-level electrode measurement, rather than an epoch-level pseudo-replicate. Electrodes are retained as separate observations.

For each sleep-stage probability, the model is `phase_mean ~ condition * phase`, with placebo as the condition reference and before as the phase reference. It includes a subject random intercept and an electrode variance component specified as `0 + C(electrode)`. There is no epoch intercept because epochs have already been averaged within phase.

With these references, the selected interactions are exactly `(After − Before)_DMT − (After − Before)_placebo` and `(Late − Before)_DMT − (Late − Before)_placebo`. The 90-second tolerance is the primary analysis; 0, 15, 30, 60, and 120 seconds evaluate sensitivity. Benjamini–Hochberg correction is applied separately within each tolerance across the 30 stager × sleep-stage × contrast tests.

## 3. Load results and average by phase

In [9]:
def load_phase_means(tolerance):
    files = sorted(RUN_DIRS[tolerance].joinpath('recordings').glob('*_inner_hypnodensities.parquet'))
    if not files:
        raise FileNotFoundError(f'No persisted hypnodensity files found for {tolerance} seconds.')
    frames = [pd.read_parquet(file) for file in files]
    table = pd.concat(frames, ignore_index=True)
    table = table.loc[table['stager'].astype(str).eq(STAGER)].copy()
    table['phase'] = table['experimental_label'].astype(str)
    table = table.loc[table['phase'].isin(PHASES)].copy()
    table['condition'] = pd.Categorical(table['condition'], categories=CONDITIONS)
    table['phase'] = pd.Categorical(table['phase'], categories=PHASES, ordered=True)
    keys = ['subject', 'condition', 'phase', 'electrode', 'stager']
    means = table.groupby(keys, observed=True, dropna=False)[list(PROBABILITIES)].mean().reset_index()
    means.insert(0, 'tolerance_seconds', tolerance)
    return means

phase_means = pd.concat([load_phase_means(tolerance) for tolerance in TOLERANCES], ignore_index=True)
phase_means['condition'] = pd.Categorical(phase_means['condition'], categories=CONDITIONS)
phase_means['phase'] = pd.Categorical(phase_means['phase'], categories=PHASES, ordered=True)
display(phase_means.groupby('tolerance_seconds', observed=True).size().rename('phase_mean_observations').to_frame())

,phase_mean_observations
tolerance_seconds,
0,3200
15,3200
30,3200
60,3200
90,3200
120,3200


## 4. Fit the LMEs

In [5]:
def benjamini_hochberg(p_values):
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(values.shape, np.nan)
    finite = np.flatnonzero(np.isfinite(values))
    if finite.size:
        order = finite[np.argsort(values[finite])]
        ranked = values[order] * finite.size / np.arange(1, finite.size + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted

def interaction_term(params, phase):
    matches = [name for name in params.index if '[T.DMT]' in name and f'[T.{phase}]' in name]
    if len(matches) != 1:
        raise KeyError(f'Expected one DMT × {phase} interaction, found {matches}.')
    return matches[0]

rows = []
for tolerance in TOLERANCES:
    subset_tolerance = phase_means.loc[phase_means['tolerance_seconds'].eq(tolerance)]
    for stage, probability in zip(STAGES, PROBABILITIES):
        data = subset_tolerance[[
            'subject', 'condition', 'phase', 'electrode', probability
        ]].dropna(subset=[probability]).rename(columns={probability: 'phase_mean'}).copy()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            model = smf.mixedlm(
                MODEL_FORMULA, data=data, groups=data['subject'],
                vc_formula={'electrode': '0 + C(electrode)'}, re_formula='1'
            )
            fit = model.fit(reml=False, method='lbfgs', maxiter=400, disp=False)
        confidence = fit.conf_int()
        for phase, label in CONTRAST_LABELS.items():
            term = interaction_term(fit.fe_params, phase)
            rows.append({
                'tolerance_seconds': tolerance, 'stager': 'GSSC', 'sleep_stage': stage,
                'contrast': label, 'estimate': float(fit.fe_params[term]),
                'standard_error': float(fit.bse_fe[term]),
                'ci95_low': float(confidence.loc[term, 0]),
                'ci95_high': float(confidence.loc[term, 1]),
                'z_value': float(fit.tvalues[term]), 'p_value': float(fit.pvalues[term]),
                'n_observations': int(len(data)), 'n_subjects': int(data['subject'].nunique()),
                'converged': bool(fit.converged),
            })

lme_interactions = pd.DataFrame(rows)
lme_interactions['p_fdr_bh'] = (
    lme_interactions.groupby('tolerance_seconds', sort=False)['p_value']
    .transform(benjamini_hochberg)
)
lme_interactions = lme_interactions[[
    'tolerance_seconds', 'stager', 'sleep_stage', 'contrast', 'estimate',
    'standard_error', 'ci95_low', 'ci95_high', 'z_value', 'p_value',
    'p_fdr_bh', 'n_observations', 'n_subjects', 'converged'
]]

## 5. Result tables and persistence

In [7]:
def format_p(value):
    if pd.isna(value):
        return ''
    return f'{value:.2e}' if abs(value) < 1e-4 else f'{value:.4f}'

def dark_table_style(styler):
    return (styler.set_properties(**{'background-color': '#1f2937', 'color': '#f3f4f6'})
                  .set_table_styles([
                      {'selector': 'th', 'props': [('background-color', '#111827'), ('color', '#f9fafb'), ('font-weight', '600')]},
                      {'selector': 'td', 'props': [('border-color', '#374151')]},
                  ]))

def style_results(table):
    numeric = ['estimate', 'standard_error', 'ci95_low', 'ci95_high', 'z_value']
    styler = dark_table_style(table.style).format({column: '{:.4f}' for column in numeric})
    for column in ('p_value', 'p_fdr_bh'):
        styler = styler.format({column: format_p})
    return styler.apply(
        lambda row: ['background-color: #365314; color: #ecfccb' if row['p_fdr_bh'] < 0.05 else '' for _ in row],
        axis=1,
    )

ordered = ['stager', 'sleep_stage', 'contrast', 'tolerance_seconds']
primary_90s = (lme_interactions.loc[lme_interactions['tolerance_seconds'].eq(90)]
               .sort_values(['stager', 'sleep_stage', 'contrast']).reset_index(drop=True))
comparison = lme_interactions.sort_values(ordered).reset_index(drop=True)
summary = comparison.pivot(index=['stager', 'sleep_stage', 'contrast'], columns='tolerance_seconds', values=['estimate', 'p_fdr_bh'])
summary.columns = [f'{metric}_{int(tolerance):03d}s' for metric, tolerance in summary.columns]
summary = summary.reset_index().sort_values(['stager', 'sleep_stage', 'contrast'])

display(style_results(primary_90s))
display(style_results(comparison))
summary_styler = dark_table_style(summary.style)
summary_styler = summary_styler.format({column: format_p for column in summary.columns if column.startswith('p_fdr_bh_')})
summary_styler = summary_styler.format({column: '{:.4f}' for column in summary.columns if column.startswith('estimate_')})
summary_styler = summary_styler.apply(lambda row: [
    'background-color: #365314; color: #ecfccb'
    if column.startswith('p_fdr_bh_') and pd.notna(value) and value < 0.05 else ''
    for column, value in zip(row.index, row)
], axis=1)
display(summary_styler)

# RESULTS_DIR.mkdir(parents=True)
phase_means.to_parquet(RESULTS_DIR / 'phase_means.parquet', index=False)
lme_interactions.to_csv(RESULTS_DIR / 'lme_interactions_all_tolerances.csv', index=False)
primary_90s.to_csv(RESULTS_DIR / 'lme_interactions_primary_90s.csv', index=False)
summary.to_csv(RESULTS_DIR / 'lme_interactions_comparison.csv', index=False)
display(f'Saved results to {RESULTS_DIR}')

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
0,90,GSSC,N1,After difference-in-differences,0.054287,0.008025,0.038559,0.070015,6.764979,0.000000,2.67e-11,3200,18,True
1,90,GSSC,N1,Late difference-in-differences,-0.054658,0.007823,-0.069991,-0.039325,-6.986793,0.000000,7.03e-12,3200,18,True
2,90,GSSC,N2,After difference-in-differences,-0.003029,0.006942,-0.016635,0.010576,-0.436421,0.662531,0.6625,3200,18,True
3,90,GSSC,N2,Late difference-in-differences,-0.050036,0.006767,-0.063299,-0.036774,-7.394484,0.000000,4.73e-13,3200,18,True
4,90,GSSC,N3,After difference-in-differences,-0.001066,0.000198,-0.001455,-0.000677,-5.371318,0.000000,1.12e-07,3200,18,False
5,90,GSSC,N3,Late difference-in-differences,-0.000184,0.000193,-0.000563,0.000196,-0.949472,0.342380,0.3804,3200,18,False
6,90,GSSC,R,After difference-in-differences,0.023459,0.002501,0.018558,0.028360,9.381296,0.000000,6.52e-20,3200,18,True
7,90,GSSC,R,Late difference-in-differences,0.005983,0.002439,0.001203,0.010763,2.453058,0.014165,0.0177,3200,18,True
8,90,GSSC,W,After difference-in-differences,-0.073662,0.013298,-0.099726,-0.047598,-5.539202,0.000000,5.06e-08,3200,18,True
9,90,GSSC,W,Late difference-in-differences,0.098690,0.012964,0.073281,0.124099,7.612693,0.000000,1.34e-13,3200,18,True


,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
0,0,GSSC,N1,After difference-in-differences,0.070001,0.008840,0.052676,0.087326,7.918997,0.000000,1.20e-14,3200,18,True
1,15,GSSC,N1,After difference-in-differences,0.070358,0.008718,0.053270,0.087446,8.070043,0.000000,3.51e-15,3200,18,True
2,30,GSSC,N1,After difference-in-differences,0.090612,0.008477,0.073998,0.107226,10.689484,0.000000,1.14e-25,3200,18,True
3,60,GSSC,N1,After difference-in-differences,0.058507,0.008602,0.041648,0.075367,6.801632,0.000000,2.59e-11,3200,18,True
4,90,GSSC,N1,After difference-in-differences,0.054287,0.008025,0.038559,0.070015,6.764979,0.000000,2.67e-11,3200,18,True
5,120,GSSC,N1,After difference-in-differences,0.052435,0.007924,0.036905,0.067964,6.617557,0.000000,7.30e-11,3200,18,True
6,0,GSSC,N1,Late difference-in-differences,-0.043060,0.008617,-0.059949,-0.026170,-4.996990,0.000001,1.46e-06,3200,18,True
7,15,GSSC,N1,Late difference-in-differences,-0.037100,0.008499,-0.053757,-0.020442,-4.365211,0.000013,2.54e-05,3200,18,True
8,30,GSSC,N1,Late difference-in-differences,-0.034284,0.008263,-0.050480,-0.018088,-4.148905,0.000033,6.68e-05,3200,18,True
9,60,GSSC,N1,Late difference-in-differences,-0.071054,0.008386,-0.087490,-0.054619,-8.473380,0.000000,1.19e-16,3200,18,True


,stager,sleep_stage,contrast,estimate_000s,estimate_015s,estimate_030s,estimate_060s,estimate_090s,estimate_120s,p_fdr_bh_000s,p_fdr_bh_015s,p_fdr_bh_030s,p_fdr_bh_060s,p_fdr_bh_090s,p_fdr_bh_120s
0,GSSC,N1,After difference-in-differences,0.0700,0.0704,0.0906,0.0585,0.0543,0.0524,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,GSSC,N1,Late difference-in-differences,-0.0431,-0.0371,-0.0343,-0.0711,-0.0547,-0.0583,0.000001,0.000025,0.000067,0.000000,0.000000,0.000000
2,GSSC,N2,After difference-in-differences,0.0016,0.0007,-0.0000,-0.0145,-0.0030,-0.0100,0.804538,0.914461,0.994770,0.038098,0.662531,0.198830
3,GSSC,N2,Late difference-in-differences,-0.0211,-0.0249,-0.0236,-0.0313,-0.0500,-0.0542,0.001310,0.000121,0.000285,0.000003,0.000000,0.000000
4,GSSC,N3,After difference-in-differences,-0.0001,-0.0001,-0.0002,-0.0010,-0.0011,-0.0013,0.303214,0.153508,0.015413,0.000002,0.000000,0.000000
5,GSSC,N3,Late difference-in-differences,-0.0000,-0.0001,-0.0001,-0.0000,-0.0002,-0.0002,0.720520,0.350757,0.594956,0.859649,0.380423,0.326614
6,GSSC,R,After difference-in-differences,0.0143,0.0150,0.0182,0.0171,0.0235,0.0191,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,GSSC,R,Late difference-in-differences,0.0005,0.0011,0.0000,-0.0003,0.0060,0.0007,0.804538,0.556895,0.994770,0.859649,0.017706,0.726297
8,GSSC,W,After difference-in-differences,-0.0859,-0.0860,-0.1087,-0.0602,-0.0737,-0.0603,0.000000,0.000000,0.000000,0.000017,0.000000,0.000007
9,GSSC,W,Late difference-in-differences,0.0637,0.0609,0.0579,0.1026,0.0987,0.1118,0.000003,0.000007,0.000019,0.000000,0.000000,0.000000


'Saved results to /home/rherzoga/Documentos/Projects/Timmerman/dmt_hypnodensities/outputs/gap_tolerance_lme_v1'

In [12]:
primary_0 = lme_interactions.loc[
    lme_interactions["tolerance_seconds"].eq(0)
].copy()

primary_0_after = primary_0.loc[
    primary_0["contrast"].eq("After difference-in-differences")
].copy()

primary_0_late = primary_0.loc[
    primary_0["contrast"].eq("Late difference-in-differences")
].copy()


display("0-second analysis — After difference-in-differences")
display(style_results(primary_0_after))

display("0-second analysis — Late difference-in-differences")
display(style_results(primary_0_late))

'0-second analysis — After difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
0,0,GSSC,W,After difference-in-differences,-0.085907,0.013479,-0.112326,-0.059487,-6.373180,0.000000,6.17e-10,3200,18,True
2,0,GSSC,N1,After difference-in-differences,0.070001,0.008840,0.052676,0.087326,7.918997,0.000000,1.20e-14,3200,18,True
4,0,GSSC,N2,After difference-in-differences,0.001596,0.006447,-0.011041,0.014232,0.247479,0.804538,0.8045,3200,18,True
6,0,GSSC,N3,After difference-in-differences,-0.000081,0.000065,-0.000207,0.000046,-1.247403,0.212250,0.3032,3200,18,True
8,0,GSSC,R,After difference-in-differences,0.014309,0.001651,0.011073,0.017545,8.665777,0.000000,4.48e-17,3200,18,True


'0-second analysis — Late difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
1,0,GSSC,W,Late difference-in-differences,0.063659,0.013140,0.037905,0.089413,4.844705,0.000001,2.54e-06,3200,18,True
3,0,GSSC,N1,Late difference-in-differences,-0.043060,0.008617,-0.059949,-0.026170,-4.996990,0.000001,1.46e-06,3200,18,True
5,0,GSSC,N2,Late difference-in-differences,-0.021102,0.006285,-0.033420,-0.008784,-3.357596,0.000786,0.0013,3200,18,True
7,0,GSSC,N3,Late difference-in-differences,-0.000035,0.000063,-0.000159,0.000088,-0.558628,0.576416,0.7205,3200,18,True
9,0,GSSC,R,Late difference-in-differences,0.000491,0.001610,-0.002664,0.003646,0.305069,0.760314,0.8045,3200,18,True


In [ ]:
primary_after = primary_90s.loc[
    primary_90s["contrast"].eq("After difference-in-differences")
].copy()

primary_late = primary_90s.loc[
    primary_90s["contrast"].eq("Late difference-in-differences")
].copy()

all_after = comparison.loc[
    comparison["contrast"].eq("After difference-in-differences")
].copy()

all_late = comparison.loc[
    comparison["contrast"].eq("Late difference-in-differences")
].copy()


display("Primary analysis — After difference-in-differences")
display(style_results(primary_after))

display("Primary analysis — Late difference-in-differences")
display(style_results(primary_late))

display("All tolerances — After difference-in-differences")
display(style_results(all_after))

display("All tolerances — Late difference-in-differences")
display(style_results(all_late))

'Primary analysis — After difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
0,90,GSSC,N1,After difference-in-differences,0.054287,0.008025,0.038559,0.070015,6.764979,0.000000,2.67e-11,3200,18,True
2,90,GSSC,N2,After difference-in-differences,-0.003029,0.006942,-0.016635,0.010576,-0.436421,0.662531,0.6625,3200,18,True
4,90,GSSC,N3,After difference-in-differences,-0.001066,0.000198,-0.001455,-0.000677,-5.371318,0.000000,1.12e-07,3200,18,False
6,90,GSSC,R,After difference-in-differences,0.023459,0.002501,0.018558,0.028360,9.381296,0.000000,6.52e-20,3200,18,True
8,90,GSSC,W,After difference-in-differences,-0.073662,0.013298,-0.099726,-0.047598,-5.539202,0.000000,5.06e-08,3200,18,True


'Primary analysis — Late difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
1,90,GSSC,N1,Late difference-in-differences,-0.054658,0.007823,-0.069991,-0.039325,-6.986793,0.000000,7.03e-12,3200,18,True
3,90,GSSC,N2,Late difference-in-differences,-0.050036,0.006767,-0.063299,-0.036774,-7.394484,0.000000,4.73e-13,3200,18,True
5,90,GSSC,N3,Late difference-in-differences,-0.000184,0.000193,-0.000563,0.000196,-0.949472,0.342380,0.3804,3200,18,False
7,90,GSSC,R,Late difference-in-differences,0.005983,0.002439,0.001203,0.010763,2.453058,0.014165,0.0177,3200,18,True
9,90,GSSC,W,Late difference-in-differences,0.098690,0.012964,0.073281,0.124099,7.612693,0.000000,1.34e-13,3200,18,True


'All tolerances — After difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
0,0,GSSC,N1,After difference-in-differences,0.070001,0.008840,0.052676,0.087326,7.918997,0.000000,1.20e-14,3200,18,True
1,15,GSSC,N1,After difference-in-differences,0.070358,0.008718,0.053270,0.087446,8.070043,0.000000,3.51e-15,3200,18,True
2,30,GSSC,N1,After difference-in-differences,0.090612,0.008477,0.073998,0.107226,10.689484,0.000000,1.14e-25,3200,18,True
3,60,GSSC,N1,After difference-in-differences,0.058507,0.008602,0.041648,0.075367,6.801632,0.000000,2.59e-11,3200,18,True
4,90,GSSC,N1,After difference-in-differences,0.054287,0.008025,0.038559,0.070015,6.764979,0.000000,2.67e-11,3200,18,True
5,120,GSSC,N1,After difference-in-differences,0.052435,0.007924,0.036905,0.067964,6.617557,0.000000,7.30e-11,3200,18,True
12,0,GSSC,N2,After difference-in-differences,0.001596,0.006447,-0.011041,0.014232,0.247479,0.804538,0.8045,3200,18,True
13,15,GSSC,N2,After difference-in-differences,0.000691,0.006434,-0.011919,0.013301,0.107413,0.914461,0.9145,3200,18,False
14,30,GSSC,N2,After difference-in-differences,-0.000042,0.006451,-0.012685,0.012600,-0.006555,0.994770,0.9948,3200,18,True
15,60,GSSC,N2,After difference-in-differences,-0.014508,0.006705,-0.027650,-0.001367,-2.163814,0.030479,0.0381,3200,18,True


'All tolerances — Late difference-in-differences'

,tolerance_seconds,stager,sleep_stage,contrast,estimate,standard_error,ci95_low,ci95_high,z_value,p_value,p_fdr_bh,n_observations,n_subjects,converged
6,0,GSSC,N1,Late difference-in-differences,-0.043060,0.008617,-0.059949,-0.026170,-4.996990,0.000001,1.46e-06,3200,18,True
7,15,GSSC,N1,Late difference-in-differences,-0.037100,0.008499,-0.053757,-0.020442,-4.365211,0.000013,2.54e-05,3200,18,True
8,30,GSSC,N1,Late difference-in-differences,-0.034284,0.008263,-0.050480,-0.018088,-4.148905,0.000033,6.68e-05,3200,18,True
9,60,GSSC,N1,Late difference-in-differences,-0.071054,0.008386,-0.087490,-0.054619,-8.473380,0.000000,1.19e-16,3200,18,True
10,90,GSSC,N1,Late difference-in-differences,-0.054658,0.007823,-0.069991,-0.039325,-6.986793,0.000000,7.03e-12,3200,18,True
11,120,GSSC,N1,Late difference-in-differences,-0.058255,0.007725,-0.073395,-0.043116,-7.541604,0.000000,1.16e-13,3200,18,True
18,0,GSSC,N2,Late difference-in-differences,-0.021102,0.006285,-0.033420,-0.008784,-3.357596,0.000786,0.0013,3200,18,True
19,15,GSSC,N2,Late difference-in-differences,-0.024884,0.006272,-0.037176,-0.012592,-3.967791,0.000073,0.0001,3200,18,False
20,30,GSSC,N2,Late difference-in-differences,-0.023631,0.006288,-0.035955,-0.011307,-3.758156,0.000171,0.0003,3200,18,True
21,60,GSSC,N2,Late difference-in-differences,-0.031259,0.006536,-0.044070,-0.018449,-4.782595,0.000002,2.88e-06,3200,18,True


In [10]:
epoch_counts = []

for tolerance in TOLERANCES:
    files = sorted(
        RUN_DIRS[tolerance]
        .joinpath("recordings")
        .glob("*_inner_hypnodensities.parquet")
    )

    raw = pd.concat(
        [pd.read_parquet(file) for file in files],
        ignore_index=True,
    )

    raw = raw.loc[
        raw["stager"].astype(str).eq(STAGER)
        & raw["experimental_label"].astype(str).isin(PHASES)
    ].copy()

    unique_epochs = raw.drop_duplicates(
        subset=[
            "subject",
            "condition",
            "recording_id",
            "block_id",
            "epoch",
        ]
    )

    counts = (
        unique_epochs
        .groupby(["condition", "subject"], observed=True)
        .size()
        .rename("n_epochs")
        .reset_index()
        .assign(tolerance_seconds=tolerance)
    )

    epoch_counts.append(counts)


epoch_counts = pd.concat(epoch_counts, ignore_index=True)

epoch_counts_table = (
    epoch_counts
    .pivot(
        index=["condition", "subject"],
        columns="tolerance_seconds",
        values="n_epochs",
    )
    .fillna(0)
    .astype(int)
    .rename(columns=lambda tolerance: f"{int(tolerance):03d}s")
    .reset_index()
    .sort_values(["condition", "subject"])
    .reset_index(drop=True)
)

display(epoch_counts_table)

tolerance_seconds,condition,subject,000s,015s,030s,060s,090s,120s
0,DMT,P07,18,18,18,18,18,18
1,DMT,P08,32,32,32,32,32,32
2,DMT,P09,21,21,21,21,21,21
3,DMT,P10,35,35,35,35,35,35
4,DMT,P11,5,5,5,5,5,5
5,DMT,P12,4,4,4,4,4,4
6,DMT,P13,15,15,15,15,15,15
7,DMT,P14,6,6,6,6,6,6
8,DMT,P15,24,24,24,24,24,24
9,DMT,P17,41,41,41,41,41,41


In [11]:
block_counts = []

for tolerance in TOLERANCES:
    files = sorted(
        RUN_DIRS[tolerance]
        .joinpath("recordings")
        .glob("*_inner_hypnodensities.parquet")
    )

    raw = pd.concat(
        [pd.read_parquet(file) for file in files],
        ignore_index=True,
    )

    raw = raw.loc[
        raw["stager"].astype(str).eq(STAGER)
        & raw["experimental_label"].astype(str).isin(PHASES)
    ].copy()

    unique_blocks = raw.drop_duplicates(
        subset=[
            "subject",
            "condition",
            "recording_id",
            "block_id",
        ]
    )

    counts = (
        unique_blocks
        .groupby(["condition", "subject"], observed=True)
        .size()
        .rename("n_continuous_blocks")
        .reset_index()
        .assign(tolerance_seconds=tolerance)
    )

    block_counts.append(counts)


block_counts = pd.concat(block_counts, ignore_index=True)

block_counts_table = (
    block_counts
    .pivot(
        index=["condition", "subject"],
        columns="tolerance_seconds",
        values="n_continuous_blocks",
    )
    .fillna(0)
    .astype(int)
    .rename(columns=lambda tolerance: f"{int(tolerance):03d}s")
    .reset_index()
    .sort_values(["condition", "subject"])
    .reset_index(drop=True)
)

display(block_counts_table)

tolerance_seconds,condition,subject,000s,015s,030s,060s,090s,120s
0,DMT,P07,11,11,8,4,2,2
1,DMT,P08,25,23,17,8,8,7
2,DMT,P09,15,15,12,7,6,6
3,DMT,P10,18,17,15,9,6,5
4,DMT,P11,5,4,4,3,3,3
5,DMT,P12,4,4,3,3,2,2
6,DMT,P13,12,12,12,11,7,6
7,DMT,P14,6,4,3,3,2,2
8,DMT,P15,21,18,16,12,6,5
9,DMT,P17,27,23,17,11,8,7
